# Side notebook: minGPT, a GPT you can read in one sitting

[minGPT](https://github.com/karpathy/minGPT) by Andrej Karpathy is a complete GPT in about **300 lines of PyTorch**. Big libraries like 🤗 Transformers hide the model behind many layers of abstraction. minGPT shows everything, so it's the best place to see what's really inside a GPT.

In this notebook we **rebuild minGPT piece by piece**, explain each part, and then use it in three experiments:

1. **Sorting numbers:** a tiny GPT learns to sort a list (trains in about a minute).
2. **Shakespeare:** a character-level GPT learns to write fake Shakespeare (a few minutes).
3. **Real GPT-2:** we load OpenAI's GPT-2 weights *into our own code* and check that it gives the same predictions as 🤗 Transformers. This shows our 300 lines really are GPT-2.

### How it connects to the book

| minGPT piece | Where the book covers it |
|:--|:--|
| Self-attention, multi-head attention, feed-forward layer, layer norm, position embeddings | Chapter 3, *Transformer Anatomy* (there for an **encoder**; here a **decoder**) |
| `generate()`: greedy search, temperature, top-k sampling | Chapter 5, *Text Generation* |
| Training loop, AdamW, weight decay, gradient clipping | Chapter 10, *Training Transformers from Scratch* |
| Byte-level BPE tokenizer (`bpe.py`) | Chapters 4 and 10 (tokenizers) |

> **Runs anywhere:** this notebook is self-contained, so you can upload just this one file to **Kaggle** or **Colab** and run it with nothing to install. On Kaggle, turn on **Internet** (to download the Shakespeare text and GPT-2) and a **GPU** accelerator in the notebook settings.

> **Hardware:** everything runs in a few minutes on a GPU. It also works on a CPU, just slower. If you get `CUDA out of memory`, check `nvidia-smi` and free the GPU first (e.g. `ollama stop <model>`).

> **Credits:** the code is adapted from [karpathy/minGPT](https://github.com/karpathy/minGPT) (MIT License, © 2020 Andrej Karpathy). Small changes for clarity and for current libraries are marked with `# (change)` comments.

In [ ]:
# This notebook is self-contained: it only needs torch, transformers, numpy and matplotlib,
# which are preinstalled on Kaggle and Colab (locally, use the nlpt environment). Nothing to install.
import torch
import transformers

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected: everything still works, just slower 🐢")
    print("  Kaggle: Settings → Accelerator → GPU   |   Colab: Runtime → Change runtime type → GPU")

In [ ]:
import hashlib
import math
import os
import random
import time
import urllib.request
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(3407)

## 1. The big picture

A GPT does one thing: **given some tokens, it predicts the next token**. Everything else is details. Here is how data flows through it:

```
token ids                         shape (B, T)
   │
   ├── wte: token embedding    ┐
   ├── wpe: position embedding ┘  added together → x   shape (B, T, C)
   │
   ▼
Block 1 → Block 2 → ... → Block N     every block does:
   │                                     x = x + Attention(LayerNorm(x))
   │                                     x = x + MLP(LayerNorm(x))
   ▼
ln_f: final LayerNorm
   ▼
lm_head: Linear(C → vocab_size)  →  logits   shape (B, T, vocab_size)
```

The letters used everywhere in the code:

- **B** = batch size (how many sequences at once)
- **T** = time = sequence length (how many tokens per sequence)
- **C** = channels = embedding size (`n_embd`), the length of the vector that represents each token

The output **logits** contain, for *every position*, a score for *every token in the vocabulary*: "how likely is this token to come next?".

### How this differs from the encoder in Chapter 3
Chapter 3 built a BERT-style **encoder**: every token can look at every other token. A GPT is a **decoder**: each token can only look at itself and the tokens **before** it. That restriction (the *causal mask*) is what lets it learn to predict the next token without cheating, by looking at the answer.

## 2. The configuration

minGPT stores its hyperparameters in a small config object. We use a Python `dataclass`, which is simpler. The original uses its own `CfgNode` class.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int          # how many different tokens exist
    block_size: int          # the longest sequence the model can see (its "context window")
    n_layer: int = 3         # how many transformer blocks are stacked
    n_head: int = 3          # how many attention heads per block
    n_embd: int = 48         # the size C of each token's vector
    embd_pdrop: float = 0.1  # dropout after the embeddings
    resid_pdrop: float = 0.1 # dropout on the residual branches
    attn_pdrop: float = 0.1  # dropout on the attention weights

# The same presets as minGPT. The real GPT-2 sizes, plus tiny made-up ones for quick experiments.
MODEL_PRESETS = {
    "gpt2":        dict(n_layer=12, n_head=12, n_embd=768),   # 124M parameters
    "gpt2-medium": dict(n_layer=24, n_head=16, n_embd=1024),  # 350M
    "gpt2-large":  dict(n_layer=36, n_head=20, n_embd=1280),  # 774M
    "gpt2-xl":     dict(n_layer=48, n_head=25, n_embd=1600),  # 1558M
    "gpt-mini":    dict(n_layer=6,  n_head=6,  n_embd=192),
    "gpt-micro":   dict(n_layer=4,  n_head=4,  n_embd=128),
    "gpt-nano":    dict(n_layer=3,  n_head=3,  n_embd=48),
}

def make_config(model_type, vocab_size, block_size, **overrides):
    return GPTConfig(vocab_size=vocab_size, block_size=block_size,
                     **{**MODEL_PRESETS[model_type], **overrides})

make_config("gpt-nano", vocab_size=10, block_size=8)

## 3. The activation function: GELU

Inside each block there is a small feed-forward network (the "MLP") that needs a non-linear activation. GPT uses **GELU**, a smooth version of ReLU. minGPT writes out the formula used by the original GPT-2 code (a fast *tanh approximation*). PyTorch has the same thing built in as `F.gelu(x, approximate="tanh")`.

In [ ]:
class NewGELU(nn.Module):
    # GELU as used in Google BERT and OpenAI GPT. Paper: https://arxiv.org/abs/1606.08415
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))

x = torch.linspace(-4, 4, 200)
print("Same as PyTorch's built-in version:", torch.allclose(NewGELU()(x), F.gelu(x, approximate="tanh"), atol=1e-6))

plt.plot(x, F.relu(x), label="ReLU")
plt.plot(x, NewGELU()(x), label="GELU")
plt.legend(); plt.title("GELU is a smooth ReLU: small negative values are not cut to exactly 0")
plt.show()

## 4. Causal self-attention, the heart of the model

Self-attention lets each token **collect information from other tokens**. For every token we compute three vectors with linear layers:

- **Query (q):** "what am I looking for?"
- **Key (k):** "what do I contain?"
- **Value (v):** "what will I give you if you pick me?"

Then:

1. **Scores:** compare every query with every key using a dot product: `q @ kᵀ`. A high score means "this token is relevant to me". We divide by `√(head size)` so the numbers don't get too big (Chapter 3 explains why).
2. **Causal mask:** set the scores for *future* tokens to `-inf`, so a token can't look ahead.
3. **Softmax:** turn each row of scores into weights that add up to 1.
4. **Weighted sum:** each token's output is the weighted average of the **values**.

### Multi-head attention
Instead of one big attention, the model runs `n_head` smaller ones in parallel. Each head works on a slice of size `hs = C / n_head` and can learn to look for something different, e.g. one head for "the previous token" and another for "a matching bracket". minGPT computes q, k and v **for all heads in one go** with a single linear layer `c_attn` (C → 3C), then reshapes the result so the heads become an extra dimension: `(B, T, C) → (B, n_head, T, hs)`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # regularization
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)
        # causal mask: a lower-triangular matrix of ones, so each position only sees the past
        # (change) persistent=False: the mask isn't a learned weight, so it's left out of the state_dict.
        # Recent 🤗 Transformers GPT-2 checkpoints don't store it either, which makes loading GPT-2 weights simpler.
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size), persistent=False)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.last_att = None    # (change) we keep the last attention weights so we can plot them

    def forward(self, x):
        B, T, C = x.size()  # batch size, sequence length, embedding size (n_embd)

        # compute q, k, v for all heads, then move the head dimension forward: (B, T, C) -> (B, nh, T, hs)
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)

        # attention scores: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))  # hide the future
        att = F.softmax(att, dim=-1)
        self.last_att = att.detach()
        att = self.attn_dropout(att)
        y = att @ v  # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # put the heads back side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

### Let's watch the shapes

The fastest way to understand attention is to follow the tensor shapes. Here's one forward pass with a tiny configuration (C = 12, 3 heads → head size 4), done step by step:

In [ ]:
cfg = GPTConfig(vocab_size=10, block_size=8, n_head=3, n_embd=12)
attn = CausalSelfAttention(cfg).eval()   # eval() turns dropout off

B, T, C = 2, 5, cfg.n_embd
x = torch.randn(B, T, C)
print("input x               ", tuple(x.shape), "  (B, T, C)")

qkv = attn.c_attn(x)
print("c_attn(x)             ", tuple(qkv.shape), "  (B, T, 3C): q, k and v glued together")
q, k, v = qkv.split(C, dim=2)
q = q.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2)
k = k.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2)
print("q (and k, v) per head ", tuple(q.shape), "  (B, n_head, T, head_size)")

scores = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))
print("scores q @ kᵀ          ", tuple(scores.shape), "  (B, n_head, T, T): every token vs every token")

y = attn(x)
print("output                ", tuple(y.shape), "  (B, T, C): same shape as the input!")

Note the last line: **attention doesn't change the shape**. That's why we can stack many blocks on top of each other.

### Seeing the causal mask
Left: the mask (1 = allowed to look, 0 = hidden). Right: the attention weights of head 0 for the first sequence. Everything above the diagonal is exactly 0, so token 3 can look at tokens 0–3 but never at token 4. Each row sums to 1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(attn.bias[0, 0, :T, :T], cmap="Greys")
axes[0].set_title("causal mask")
axes[1].imshow(attn.last_att[0, 0], cmap="viridis")
axes[1].set_title("attention weights (head 0)")
for ax in axes:
    ax.set_xlabel("key position (looked at)"); ax.set_ylabel("query position (looking)")
plt.tight_layout(); plt.show()
print("row sums:", attn.last_att[0, 0].sum(-1))

### The modern shortcut
Today PyTorch has a fused, much faster version of exactly these steps: `F.scaled_dot_product_attention`. It can use *FlashAttention* on recent GPUs. minGPT writes everything out by hand so you can read it, but the result is the same:

In [ ]:
qkv = attn.c_attn(x)
q, k, v = [t.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2) for t in qkv.split(C, dim=2)]
y_fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
y_fast = attn.c_proj(y_fast.transpose(1, 2).reshape(B, T, C))
print("Same result as our hand-written attention:", torch.allclose(y_fast, attn(x), atol=1e-6))

## 5. The transformer block

A block combines attention with a small feed-forward network (the **MLP**: Linear C → 4C, GELU, Linear 4C → C). Attention lets tokens *talk to each other*; the MLP lets each token *think on its own* about what it collected.

Two details matter a lot:

- **Residual connections** (`x = x + ...`): each sub-layer only *adds* a correction to `x` instead of replacing it. You can picture `x` as a "residual stream" flowing up through the network, with every block reading from it and writing back into it. This is what makes deep networks trainable.
- **Pre-LayerNorm:** the LayerNorm is applied *before* attention and MLP, not after. Chapter 3 discusses pre- vs. post-layer normalization; GPT-2 uses pre-LN because it trains more stably.

In [ ]:
class Block(nn.Module):
    # an unassuming Transformer block
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.ModuleDict(dict(
            c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd),
            c_proj  = nn.Linear(4 * config.n_embd, config.n_embd),
            act     = NewGELU(),
            dropout = nn.Dropout(config.resid_pdrop),
        ))

    def mlpf(self, x):
        # (change) a method instead of the original lambda, same computation
        m = self.mlp
        return m.dropout(m.c_proj(m.act(m.c_fc(x))))

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # tokens talk to each other
        x = x + self.mlpf(self.ln_2(x))   # each token thinks on its own
        return x

## 6. The full GPT model

Now we put everything together:

- **`wte` (word token embedding):** a lookup table, one vector of size C per token id.
- **`wpe` (word position embedding):** one vector per *position* 0 … block_size-1. Attention by itself doesn't know the order of the tokens, so the position vectors are added to tell it. GPT *learns* these vectors; the original Transformer used fixed sine waves.
- **`h`:** the stack of `n_layer` blocks.
- **`ln_f` + `lm_head`:** a final LayerNorm, then a linear layer that turns each C-vector into one score per vocabulary token.

Three more things are worth knowing:

- **Weight initialization:** weights start as small random numbers (std 0.02, like GPT-2). The output projections of each block (`c_proj`) are scaled down by `√(2·n_layer)`, because every block adds to the residual stream and we don't want the sum to blow up as we stack more blocks.
- **The loss:** `forward` can also take `targets` and compute the cross-entropy loss. Targets equal to **-1 are ignored** (`ignore_index=-1`), which the sorting experiment uses to say "don't grade these positions".
- **`configure_optimizers`:** weight decay (a pull towards 0 that prevents overfitting) is only applied to the big matrices of the `Linear` layers, never to biases, LayerNorm or embeddings.

In [ ]:
class GPT(nn.Module):
    # GPT Language Model

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None and config.block_size is not None
        self.config = config
        self.block_size = config.block_size

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.embd_pdrop),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # init all weights, and apply a special scaled init to the residual projections, per GPT-2 paper
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        # report number of parameters (not counting lm_head, as in minGPT)
        n_params = sum(p.numel() for p in self.transformer.parameters())
        print(f"number of parameters: {n_params / 1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def configure_optimizers(self, learning_rate, weight_decay, betas):
        # Split the parameters into two groups: with and without weight decay
        decay, no_decay = set(), set()
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters(recurse=False):
                fpn = f"{mn}.{pn}" if mn else pn  # full parameter name
                if pn.endswith("bias"):
                    no_decay.add(fpn)                       # biases: no decay
                elif pn.endswith("weight") and isinstance(m, nn.Linear):
                    decay.add(fpn)                          # big matrices: decay
                elif pn.endswith("weight") and isinstance(m, (nn.LayerNorm, nn.Embedding)):
                    no_decay.add(fpn)                       # norms and embeddings: no decay
        param_dict = dict(self.named_parameters())
        assert not (decay & no_decay) and not (param_dict.keys() - (decay | no_decay))
        optim_groups = [
            {"params": [param_dict[pn] for pn in sorted(decay)], "weight_decay": weight_decay},
            {"params": [param_dict[pn] for pn in sorted(no_decay)], "weight_decay": 0.0},
        ]
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        assert t <= self.block_size, f"Cannot forward sequence of length {t}, block size is only {self.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device).unsqueeze(0)  # shape (1, t)

        tok_emb = self.transformer.wte(idx)  # (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos)  # (1, t, n_embd), broadcast over the batch
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)             # (b, t, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, do_sample=False, top_k=None):
        # Repeatedly predict the next token and append it to the sequence
        for _ in range(max_new_tokens):
            # if the sequence is too long, keep only the last block_size tokens
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            # we only need the scores at the LAST position; temperature reshapes them
            logits = logits[:, -1, :] / temperature
            # optionally keep only the top_k most likely tokens
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            # either sample from the distribution or take the most likely token (greedy)
            if do_sample:
                idx_next = torch.multinomial(probs, num_samples=1)
            else:
                _, idx_next = torch.topk(probs, k=1, dim=-1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

### A first forward pass

Let's create a tiny untrained model and push a random batch through it. One useful sanity check: **before training, the loss should be about `ln(vocab_size)`**. The untrained model guesses every token with roughly the same probability `1/vocab_size`, and cross-entropy of a uniform guess is `-ln(1/vocab_size) = ln(vocab_size)`. If your first loss is far from this value, something is wrong with the initialization.

In [ ]:
cfg = make_config("gpt-nano", vocab_size=10, block_size=8)
tiny = GPT(cfg)

idx = torch.randint(0, cfg.vocab_size, (4, 8))       # 4 random sequences of 8 tokens
targets = torch.randint(0, cfg.vocab_size, (4, 8))
logits, loss = tiny(idx, targets)
print("logits shape:", tuple(logits.shape), " (B, T, vocab_size)")
print(f"initial loss: {loss.item():.3f}   expected ≈ ln(10) = {math.log(10):.3f}")

## 7. How `generate()` works

GPT creates text **one token at a time**. This loop is called *autoregressive generation*:

```
"The cat"  → model → scores for the next token → pick "sat" → "The cat sat"
"The cat sat" → model → ... → pick "on" → "The cat sat on" → ...
```

At each step `generate()`:

1. **Crops the context** to the last `block_size` tokens. The model can't see further back than its context window.
2. **Takes only the last position's logits:** the prediction for the *next* token.
3. **Divides by the temperature.** Below 1 the distribution becomes sharper (safer, more repetitive); above 1 it becomes flatter (more creative, more mistakes).
4. Optionally keeps only the **top-k** candidates, so very unlikely tokens are never picked.
5. Either **samples** (`do_sample=True`) or takes the **most likely token** (greedy).

These are exactly the decoding methods of Chapter 5. 🤗 Transformers' `model.generate()` does the same thing, with many more options and a cache that avoids recomputing old tokens (minGPT recomputes the whole sequence every step, which is simple but slower).

## 8. The training loop

minGPT's `Trainer` is short, but it contains the standard recipe for training transformers:

- **AdamW** with `betas=(0.9, 0.95)` and the weight-decay groups from `configure_optimizers`.
- **Gradient clipping** (`clip_grad_norm_` to 1.0): if one batch produces huge gradients, they're scaled down so a single bad step can't wreck the model.
- **An endless stream of random batches:** `RandomSampler(..., replacement=True)` draws examples forever, and we stop after `max_iters` steps instead of counting epochs.
- **Callbacks:** a function called after every step, e.g. to print the loss or generate a sample.

We also record the loss so we can plot it.

In [ ]:
@dataclass
class TrainConfig:
    max_iters: int = 2000
    batch_size: int = 64
    learning_rate: float = 3e-4
    betas: tuple = (0.9, 0.95)
    weight_decay: float = 0.1     # only applied to the Linear weight matrices
    grad_norm_clip: float = 1.0
    num_workers: int = 0          # 0 = load data in the main process (simplest inside notebooks)


class Trainer:
    def __init__(self, config, model, train_dataset):
        self.config = config
        self.model = model.to(device)
        self.train_dataset = train_dataset
        self.callbacks = []
        self.losses = []          # (change) keep the loss history for plotting
        self.iter_num = 0

    def add_callback(self, fn):
        self.callbacks.append(fn)

    def run(self):
        model, config = self.model, self.config
        self.optimizer = model.configure_optimizers(config.learning_rate, config.weight_decay, config.betas)
        train_loader = DataLoader(
            self.train_dataset,
            sampler=torch.utils.data.RandomSampler(self.train_dataset, replacement=True, num_samples=int(1e10)),
            batch_size=config.batch_size,
            num_workers=config.num_workers,
            pin_memory=(device == "cuda"),
        )
        model.train()
        data_iter = iter(train_loader)
        self.t0 = time.time()
        while self.iter_num < config.max_iters:
            x, y = [t.to(device) for t in next(data_iter)]

            logits, self.loss = model(x, y)        # forward
            model.zero_grad(set_to_none=True)
            self.loss.backward()                   # backward
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_norm_clip)
            self.optimizer.step()                  # update the weights

            self.losses.append(self.loss.item())
            self.iter_num += 1
            for fn in self.callbacks:
                fn(self)
        model.eval()


def plot_losses(losses, title, window=50):
    smooth = np.convolve(losses, np.ones(window) / window, mode="valid")
    plt.plot(losses, alpha=0.3, label="loss per step")
    plt.plot(range(window - 1, len(losses)), smooth, label=f"moving average ({window} steps)")
    plt.xlabel("training step"); plt.ylabel("cross-entropy loss"); plt.title(title); plt.legend()
    plt.show()

## 9. Experiment 1: teaching a GPT to sort

This is minGPT's `demo.ipynb`. The task: given 6 digits (0, 1 or 2), output them sorted.

```
input : 0 0 2 1 0 1   →   output: 0 0 0 1 1 2
```

A GPT only knows how to "continue a sequence", so we glue problem and answer together and let it learn to continue:

```
sequence :  0 0 2 1 0 1 | 0 0 0 1 1 2
x (input):  0 0 2 1 0 1   0 0 0 1 1        (everything except the last token)
y (target): - - - - - 0   0 0 1 1 2        (everything shifted left by one)
```

At every position the target is simply **the next token**. The first five targets are set to **-1**, which means "ignore". Predicting the next *unsorted* digit is impossible (they're random), so we only grade the model on the answer part. This shift-by-one trick is how *every* GPT is trained, including GPT-2 and ChatGPT.

In [ ]:
class SortDataset(Dataset):
    # E.g. for length 6:  input 0 0 2 1 0 1 -> output 0 0 0 1 1 2

    def __init__(self, split, length=6, num_digits=3):
        assert split in {"train", "test"}
        self.split = split
        self.length = length
        self.num_digits = num_digits

    def __len__(self):
        return 10000  # the examples are generated on the fly, so this number is arbitrary

    def get_vocab_size(self):
        return self.num_digits

    def get_block_size(self):
        # input + output glued together, minus 1 because the last token is never fed in
        return self.length * 2 - 1

    def __getitem__(self, idx):
        while True:
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # half of the time, prefer examples with many repeated digits (the hard cases)
            if torch.rand(1).item() < 0.5 and inp.unique().nelement() > self.length // 2:
                continue
            # 25% of all possible inputs are reserved for the test set, decided by a hash of the digits.
            # (change) minGPT used Python's hash(), which is randomized every time Python starts;
            # md5 gives the same train/test split on every run.
            h = int(hashlib.md5(bytes(inp.tolist())).hexdigest(), 16)
            if ("test" if h % 4 == 0 else "train") == self.split:
                break
        sol = torch.sort(inp)[0]
        cat = torch.cat((inp, sol), dim=0)
        x = cat[:-1].clone()
        y = cat[1:].clone()
        y[: self.length - 1] = -1   # don't grade predictions made while reading the input
        return x, y


train_dataset = SortDataset("train")
test_dataset = SortDataset("test")
x, y = train_dataset[0]
print("x (input) :", x.tolist())
print("y (target):", y.tolist())

Now a **gpt-nano** (3 layers, 3 heads, C = 48, about 0.1M parameters) and 2000 training steps:

In [ ]:
set_seed(3407)
sort_model = GPT(make_config("gpt-nano", vocab_size=train_dataset.get_vocab_size(),
                             block_size=train_dataset.get_block_size()))

trainer = Trainer(TrainConfig(max_iters=2000, learning_rate=5e-4), sort_model, train_dataset)

def print_progress(trainer):
    if trainer.iter_num % 200 == 0:
        print(f"iter {trainer.iter_num:5d} | loss {trainer.loss.item():.4f} | {time.time() - trainer.t0:.0f}s")

trainer.add_callback(print_progress)
trainer.run()
plot_losses(trainer.losses, "Sorting: training loss")

### Does it really sort?

We give the model only the 6 input digits and let `generate()` write the next 6 tokens **greedily** (always the most likely token). The answer counts as correct only if *all* 6 digits are right. The **test** split contains digit lists the model **never saw during training**, so a high test score means it learned the *rule*, not just memorized examples.

In [ ]:
def eval_split(model, dataset, max_batches=50):
    n = dataset.length
    results, mistakes_printed = [], 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0)
    for b, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        inp = x[:, :n]                    # only the problem
        sol = y[:, -n:]                   # the correct answer
        cat = model.generate(inp, n, do_sample=False)
        sol_candidate = cat[:, n:]        # what the model wrote
        correct = (sol == sol_candidate).all(1).cpu()
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed < 3:
                mistakes_printed += 1
                print(f"GPT claims that {inp[i].tolist()} sorted is {sol_candidate[i].tolist()} but it is {sol[i].tolist()}")
        if b + 1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print(f"{dataset.split} score: {int(rt.sum())}/{len(results)} = {100 * rt.mean():.2f}% correct")

sort_model.eval()
eval_split(sort_model, train_dataset)
eval_split(sort_model, test_dataset)

In [ ]:
# Try your own sequence (6 digits, each 0, 1 or 2)
inp = torch.tensor([[0, 0, 2, 1, 0, 1]], dtype=torch.long, device=device)
cat = sort_model.generate(inp, 6, do_sample=False)
print("input      :", inp[0].tolist())
print("GPT sorted :", cat[0, 6:].tolist())
print("true sorted:", torch.sort(inp[0])[0].tolist())

### What is the model looking at?

We saved the attention weights of every layer. Below is the **last layer** for the example above: one plot per head. Rows are the positions that are *predicting*; columns are the positions they *look at*. Positions 5–10 are where the answer is written. Look for heads that, while writing the answer, focus on the input digits (columns 0–5) that have the value being written. That's the model "counting" how many 0s, 1s and 2s there are.

In [ ]:
seq = cat[:, :-1]                          # the full sequence the model saw while writing the answer
sort_model(seq)                            # one forward pass fills last_att in every layer
att = sort_model.transformer.h[-1].attn.last_att[0].cpu()   # (n_head, T, T)
labels = [str(t) for t in seq[0].tolist()]

fig, axes = plt.subplots(1, att.size(0), figsize=(4 * att.size(0), 4))
for h, ax in enumerate(axes):
    ax.imshow(att[h], cmap="viridis")
    ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
    ax.axvline(5.5, color="white", lw=1); ax.axhline(4.5, color="white", lw=1)
    ax.set_title(f"last layer, head {h}"); ax.set_xlabel("looks at"); ax.set_ylabel("predicting from")
plt.tight_layout(); plt.show()

## 10. Experiment 2: a character-level Shakespeare

This is minGPT's `projects/chargpt`. The idea is the same as before, just with real text:

- The **vocabulary** is every *character* that appears in the text (about 65: letters, punctuation, newline...). No BPE tokenizer is needed.
- Every training example is a random chunk of `block_size + 1` characters. `x` is the chunk without its last character, `y` the chunk without its first, so again **each target is the next character**.
- The text is "Tiny Shakespeare": about 1 MB of Shakespeare plays, Karpathy's classic dataset.

In [ ]:
os.makedirs("data", exist_ok=True)
shakespeare_path = "data/tinyshakespeare.txt"
if not os.path.exists(shakespeare_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, shakespeare_path)
text = open(shakespeare_path, encoding="utf-8").read()
print(f"{len(text):,} characters. The beginning:\n")
print(text[:300])

In [ ]:
class CharDataset(Dataset):
    # Emits (x, y) chunks of characters where y is x shifted by one

    def __init__(self, data, block_size=128):
        chars = sorted(set(data))
        self.stoi = {ch: i for i, ch in enumerate(chars)}   # character -> integer
        self.itos = {i: ch for i, ch in enumerate(chars)}   # integer -> character
        self.vocab_size = len(chars)
        self.block_size = block_size
        self.data = data
        print(f"data has {len(data):,} characters, {self.vocab_size} unique")

    def get_vocab_size(self):
        return self.vocab_size

    def get_block_size(self):
        return self.block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx: idx + self.block_size + 1]   # block_size + 1 characters
        dix = [self.stoi[s] for s in chunk]
        x = torch.tensor(dix[:-1], dtype=torch.long)
        y = torch.tensor(dix[1:], dtype=torch.long)
        return x, y

    def encode(self, s):
        return torch.tensor([[self.stoi[c] for c in s]], dtype=torch.long)

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)


char_dataset = CharDataset(text, block_size=128)
print(repr(char_dataset.decode(char_dataset[0][0][:60])))

A **gpt-mini** (6 layers, 6 heads, C = 192, about 2.7M parameters). Every 500 steps we let it write a little text, so you can watch it learn: random characters → word-like shapes → real words, names and line breaks → something that looks like a play.

> On a GPU this takes a few minutes. On a CPU, lower `max_iters` (e.g. 500) to try it quickly. The text will be worse, but you'll see the idea.

In [ ]:
set_seed(3407)
char_model = GPT(make_config("gpt-mini", vocab_size=char_dataset.get_vocab_size(),
                             block_size=char_dataset.get_block_size()))

char_trainer = Trainer(TrainConfig(max_iters=3000, learning_rate=5e-4), char_model, char_dataset)

def shakespeare_progress(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter {trainer.iter_num:5d} | loss {trainer.loss.item():.4f} | {time.time() - trainer.t0:.0f}s")
    if trainer.iter_num % 500 == 0:
        trainer.model.eval()
        ctx = char_dataset.encode("O God, O God!").to(device)
        out = trainer.model.generate(ctx, 200, temperature=1.0, do_sample=True, top_k=10)[0]
        print("-" * 60); print(char_dataset.decode(out)); print("-" * 60)
        trainer.model.train()

char_trainer.add_callback(shakespeare_progress)
char_trainer.run()
plot_losses(char_trainer.losses, "Shakespeare: training loss")

### Playing with the temperature

Same prompt, three temperatures (Chapter 5 explains the math):

- **0.5:** safe and repetitive;
- **1.0:** the model's own distribution;
- **1.5:** wild, with more invented words.

In [ ]:
prompt = "ROMEO:\n"
for temperature in [0.5, 1.0, 1.5]:
    set_seed(0)
    out = char_model.generate(char_dataset.encode(prompt).to(device), 250,
                              temperature=temperature, do_sample=True)[0]
    print(f"===== temperature {temperature} =====")
    print(char_dataset.decode(out))
    print()

## 11. Experiment 3: real GPT-2 weights in our own code

Our `GPT` class has **exactly the same architecture as OpenAI's GPT-2**, so we can take the real trained weights (124M parameters) from the Hugging Face Hub and copy them into it. This is minGPT's `GPT.from_pretrained()`.

Two small details when copying:

1. OpenAI's original code (and 🤗's GPT-2) uses a layer called **`Conv1D`** instead of `nn.Linear`. It does the same math, but stores the weight matrix **transposed** (shape `(in, out)` instead of `(out, in)`). So those four weight matrices need a `.t()`.
2. **Same names on purpose:** minGPT named every layer exactly like 🤗 Transformers (`transformer.h.0.attn.c_attn.weight`, ...), so the copy is a simple loop over names.

If our code is correct, both models should produce **the same logits** for the same input. That's a much stronger test than "the text looks reasonable".

> This downloads GPT-2 small (~500 MB) the first time.

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel

def gpt_from_pretrained(model_type="gpt2"):
    # Create our own GPT and copy OpenAI's GPT-2 weights into it from the 🤗 Hub
    assert model_type in {"gpt2", "gpt2-medium", "gpt2-large", "gpt2-xl"}
    config = make_config(model_type, vocab_size=50257, block_size=1024)   # GPT-2's vocab and context size
    model = GPT(config)
    sd = model.state_dict()

    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    # (change) ignore any non-weight buffers older checkpoints may contain (the causal mask)
    keys = [k for k in sd_hf if not k.endswith((".attn.masked_bias", ".attn.bias"))]
    transposed = ["attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"]
    assert set(keys) == set(sd), f"name mismatch: {set(keys) ^ set(sd)}"
    for k in keys:
        if any(k.endswith(w) for w in transposed):
            assert sd_hf[k].shape[::-1] == sd[k].shape     # Conv1D weights are stored transposed
            with torch.no_grad():
                sd[k].copy_(sd_hf[k].t())
        else:
            assert sd_hf[k].shape == sd[k].shape
            with torch.no_grad():
                sd[k].copy_(sd_hf[k])
    return model, model_hf


gpt2_ours, gpt2_hf = gpt_from_pretrained("gpt2")
gpt2_ours.to(device).eval()
gpt2_hf.to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
prompt = "Andrej Karpathy, the"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    logits_ours, _ = gpt2_ours(ids)
    logits_hf = gpt2_hf(ids).logits

print("biggest difference between the two models' logits:", (logits_ours - logits_hf).abs().max().item())
print("same next-token prediction:", torch.equal(logits_ours[0, -1].argmax(), logits_hf[0, -1].argmax()))
print("predicted next token:", repr(tokenizer.decode(logits_ours[0, -1].argmax().item())))

The difference should be tiny (around `1e-4` or less): just floating-point rounding, because 🤗 uses a fused attention kernel. **Our 300 lines compute the same thing as GPT-2.** 🎉

Now let's generate with *our* model, using top-k sampling like minGPT's `generate.ipynb`:

In [ ]:
set_seed(3407)
x = ids.expand(5, -1)          # 5 samples in one batch
y = gpt2_ours.generate(x, max_new_tokens=25, do_sample=True, top_k=40)
for i in range(y.size(0)):
    print("-" * 80)
    print(tokenizer.decode(y[i].cpu()))

In [ ]:
# Free the GPU memory before moving on
del gpt2_hf
if device == "cuda":
    torch.cuda.empty_cache()

## 12. Bonus: the BPE tokenizer

minGPT also ships `bpe.py`, a commented copy of OpenAI's GPT-2 tokenizer. Above we used 🤗's GPT-2 tokenizer, which gives exactly the same ids. Its two main ideas:

1. **Bytes, not characters.** The text is turned into UTF-8 bytes, so any text, emoji or language can be encoded with only 256 base symbols and there are never "unknown" characters. To keep the tokens printable, each byte is mapped to a visible character by `bytes_to_unicode()`. That's why spaces show up as **`Ġ`**: byte 32 (space) is shown as `chr(256 + 32)`.
2. **Byte Pair Encoding (BPE).** Starting from single bytes, the tokenizer repeatedly merges the most frequent pairs it learned during training (`"t" + "h" → "th"`, `"th" + "e" → "the"`, ...). Common words become one token; rare words are split into pieces.

Chapter 10 trains this kind of tokenizer from scratch for Python code.

In [ ]:
from transformers.convert_slow_tokenizer import bytes_to_unicode

byte_encoder = bytes_to_unicode()
print("space (byte 32)  ->", byte_encoder[32])
print("newline (byte 10)->", byte_encoder[10])

for text_example in ["Andrej Karpathy, the", "Tokenization is fun!", "東京 🤖"]:
    ids_example = tokenizer(text_example).input_ids
    print(f"\n{text_example!r}")
    print("  tokens:", tokenizer.convert_ids_to_tokens(ids_example))
    print("  ids   :", ids_example)

## 13. Wrap-up

### minGPT ↔ 🤗 Transformers

Now that you know minGPT, you can read 🤗's GPT-2 code (`transformers/models/gpt2/modeling_gpt2.py`): it's the same model with more options.

| minGPT | 🤗 Transformers GPT-2 | What it is |
|:--|:--|:--|
| `GPT` | `GPT2LMHeadModel` | the whole model + language-modeling head |
| `transformer.wte` / `transformer.wpe` | same names | token / position embeddings |
| `Block` | `GPT2Block` | one transformer block |
| `CausalSelfAttention` | `GPT2Attention` | masked multi-head self-attention |
| `mlp` (`c_fc`, `c_proj`) | `GPT2MLP` | the feed-forward network |
| `NewGELU` | `gelu_new` activation | GELU, tanh approximation |
| `generate()` | `model.generate()` | decoding loop (🤗's adds caching, beam search, ...) |
| `Trainer` | `Trainer` / Accelerate (Chapter 10) | the training loop |

### Exercises
1. **Make sorting harder:** use `SortDataset(..., length=10, num_digits=5)`. Does gpt-nano still manage? What about gpt-micro?
2. **Remove the causal mask** (comment out the `masked_fill` line) and retrain the sorter. The training loss drops much faster. Why is that cheating, and why does generation get worse?
3. **Remove the position embeddings** (`x = self.transformer.drop(tok_emb)`). Can the model still sort? Think about what attention knows without positions.
4. **Count the parameters** of `GPT(make_config("gpt2", 50257, 1024))` by hand from the layer sizes, and check against the printed 124M.
5. **Plug in the fast attention:** replace the four lines in `CausalSelfAttention.forward` with `F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=...)` and time the Shakespeare training before and after.
6. **Train on your own text:** your notes, song lyrics, or code, by just changing the `.txt` file.

### Where to go next
- **[nanoGPT](https://github.com/karpathy/nanoGPT):** Karpathy's faster successor to minGPT, which can reproduce GPT-2 training.
- **The video ["Let's build GPT: from scratch, in code, spelled out"](https://www.youtube.com/watch?v=kCc8FmEb1nY):** Karpathy builds this model live in about 2 hours. It's an excellent companion to this notebook.
- **Chapter 10 of the book:** training a GPT-2-sized code model at scale with 🤗 Accelerate.